# NSTT-Lite — Phase 6: Whisper-small Fine-Tuning (Colab T4)

**Runtime:** `Runtime → Change runtime type → T4 GPU`

Upload this notebook to Colab, then run cells top-to-bottom.

**What this does**
1. Mount Drive and open/clone the `NSTT-Lite` repo
2. Install pinned deps from `requirements.txt`
3. Verify GPU + manifests + audio
4. (Recommended) copy data to local `/content` for fast I/O
5. Full fine-tune with TensorBoard + Drive checkpoints (resumable)

**Before you start**
- Repo should live at `MyDrive/NSTT-Lite` (or this notebook will clone it)
- `data/manifests/{train,val,test}.jsonl` and `data/processed/*.wav` must be on Drive (they are gitignored — upload/sync from your Mac if missing)


## 0) Mount Google Drive


In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)


## 1) Repo on Drive (`coursework-10phase`)

If the repo is already on Drive, this just `git pull`s. Otherwise it clones.


In [ ]:
from pathlib import Path
import os

DRIVE_ROOT = Path("/content/drive/MyDrive/NSTT-Lite")
REPO_URL = "https://github.com/Rbimochan/NSTT-Lite.git"
BRANCH = "coursework-10phase"

if (DRIVE_ROOT / ".git").exists():
    %cd {DRIVE_ROOT}
    !git fetch origin
    !git checkout {BRANCH}
    !git pull --ff-only origin {BRANCH}
else:
    DRIVE_ROOT.parent.mkdir(parents=True, exist_ok=True)
    !git clone -b {BRANCH} {REPO_URL} {DRIVE_ROOT}
    %cd {DRIVE_ROOT}

print("Repo:", DRIVE_ROOT)
!git rev-parse --abbrev-ref HEAD
!git log -1 --oneline


## 2) Install dependencies

Expect yellow dependency-conflict warnings — only stop on a red `ResolutionImpossible`.

**After this cell finishes:** `Runtime → Restart session`, then re-run from cell 3 (skip install).


In [ ]:
%cd /content/drive/MyDrive/NSTT-Lite
!pip install -q -r requirements.txt
print("Install done. Restart the runtime now, then continue from the next cell.")


## 3) GPU check + imports (run after restart)


In [ ]:
from pathlib import Path
import sys
import torch

DRIVE_ROOT = Path("/content/drive/MyDrive/NSTT-Lite")
assert DRIVE_ROOT.exists(), f"Missing repo at {DRIVE_ROOT}"
%cd {DRIVE_ROOT}
sys.path.insert(0, str(DRIVE_ROOT))

print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise SystemExit("No GPU — set Runtime → Change runtime type → T4 GPU")

import transformers, datasets, evaluate, jiwer, accelerate, tensorboard, librosa, soundfile, sklearn
print("imports OK | transformers", transformers.__version__)


## 4) Verify data on Drive

Manifests + processed WAVs must already be on Drive (not in git).
If this fails, upload `data/manifests/` and `data/processed/` from your Mac to Drive `NSTT-Lite/data/`.


In [ ]:
from pathlib import Path
import json

DRIVE_ROOT = Path("/content/drive/MyDrive/NSTT-Lite")
manifest_dir = DRIVE_ROOT / "data" / "manifests"
processed = DRIVE_ROOT / "data" / "processed"

for name in ("train.jsonl", "val.jsonl", "test.jsonl"):
    p = manifest_dir / name
    assert p.exists(), f"Missing {p}"
    n = sum(1 for _ in p.open())
    print(f"{name}: {n} rows")

train_row = json.loads((manifest_dir / "train.jsonl").readline())
sample = DRIVE_ROOT / train_row["audio_path"]
assert sample.exists(), f"Missing audio {sample} — upload data/processed to Drive"
print("sample audio OK:", sample)
print("processed wav count (may be slow on Drive): …")
# cheap existence check only
print("processed dir exists:", processed.exists())


## 5) Copy data to local disk (strongly recommended)

Drive I/O over many small WAVs is ~2 it/s. Local `/content` is much faster.
Checkpoints still write to Drive so reconnects don't lose progress.


In [ ]:
from pathlib import Path
import shutil
import os

DRIVE_ROOT = Path("/content/drive/MyDrive/NSTT-Lite")
LOCAL_ROOT = Path("/content/NSTT-Lite")

# Fresh local worktree: code symlink-ish copy via rsync of needed dirs
LOCAL_ROOT.mkdir(parents=True, exist_ok=True)

# Always refresh code + requirements from Drive
for rel in ("src", "scripts", "requirements.txt"):
    src = DRIVE_ROOT / rel
    dst = LOCAL_ROOT / rel
    if src.is_dir():
        if dst.exists():
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
    else:
        shutil.copy2(src, dst)

# Manifests (tiny)
manif_dst = LOCAL_ROOT / "data" / "manifests"
manif_dst.mkdir(parents=True, exist_ok=True)
!rsync -a --info=stats2 "{DRIVE_ROOT}/data/manifests/" "{LOCAL_ROOT}/data/manifests/"

# Processed audio (large) — skip if already present with matching count
proc_src = DRIVE_ROOT / "data" / "processed"
proc_dst = LOCAL_ROOT / "data" / "processed"
proc_dst.mkdir(parents=True, exist_ok=True)
src_n = sum(1 for p in proc_src.glob("*.wav")) if proc_src.exists() else 0
dst_n = sum(1 for p in proc_dst.glob("*.wav")) if proc_dst.exists() else 0
print(f"Drive wavs={src_n}  local wavs={dst_n}")
if src_n == 0:
    raise SystemExit("No wavs on Drive under data/processed — upload them first")
if dst_n < src_n:
    print("Copying processed audio Drive → /content (one-time per session)…")
    !rsync -a --info=progress2 "{DRIVE_ROOT}/data/processed/" "{LOCAL_ROOT}/data/processed/"
else:
    print("Local processed audio already complete — skipping copy")

(LOCAL_ROOT / "reports").mkdir(parents=True, exist_ok=True)
(LOCAL_ROOT / "models").mkdir(parents=True, exist_ok=True)
print("LOCAL_ROOT ready:", LOCAL_ROOT)


## 6) Full fine-tune (resumable)

- Writes checkpoints to **Drive** `models/whisper-small-ft/`
- Uses **local** data under `/content/NSTT-Lite`
- Re-run this cell after a disconnect — it auto-resumes when checkpoints exist


In [ ]:
from pathlib import Path
import json
import sys
from datetime import datetime, timezone

import torch

DRIVE_ROOT = Path("/content/drive/MyDrive/NSTT-Lite")
LOCAL_ROOT = Path("/content/NSTT-Lite")
OUTPUT_DIR = DRIVE_ROOT / "models" / "whisper-small-ft"  # persistent
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(LOCAL_ROOT))
from src.training import train_and_save

# Prefer auto-resume when any checkpoint-* exists
resume = any(OUTPUT_DIR.glob("checkpoint-*"))
print("Device:", {
    "cuda": torch.cuda.is_available(),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
})
print("Data root:", LOCAL_ROOT)
print("Checkpoint dir:", OUTPUT_DIR)
print("Resume:", resume)

result = train_and_save(
    project_root=LOCAL_ROOT,
    output_dir=OUTPUT_DIR,
    smoke_test=False,
    resume_from_checkpoint=True if resume else None,
)

# Persist summary on Drive
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
summary = {
    "created_at": stamp,
    "device": {
        "cuda": torch.cuda.is_available(),
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
        "torch": torch.__version__,
    },
    **result,
}
reports = DRIVE_ROOT / "reports"
reports.mkdir(parents=True, exist_ok=True)
out = reports / f"phase6_train_full_{stamp}.json"
out.write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(json.dumps(summary, indent=2))
print("Saved:", out)


## 7) TensorBoard (screenshot for coursework)


In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/drive/MyDrive/NSTT-Lite/models/whisper-small-ft/runs


## 8) Optional — local smoke test (pipeline check only)

Skip for the real run. Useful if you only want to verify the Colab wiring.


In [ ]:
# SMOKE = True  # uncomment to run
SMOKE = False
if SMOKE:
    from pathlib import Path
    import sys
    sys.path.insert(0, "/content/NSTT-Lite")
    from src.training import train_and_save
    train_and_save(
        project_root=Path("/content/NSTT-Lite"),
        output_dir=Path("/content/drive/MyDrive/NSTT-Lite/models/whisper-small-ft-smoke"),
        smoke_test=True,
    )
